# Fine-tuning LoRA — Modèle finance (production, dataset ASSAINI)

**Mission Production** — réentraînement de Phi-3.5 sur `finance_dataset_clean.json`
(dataset nettoyé : la backdoor de l'équipe précédente a été retirée).

> ⚠️ Ne JAMAIS utiliser `finance_dataset_final.json` (empoisonné). Une cellule de
> garde-fou vérifie l'absence du trigger avant l'entraînement.

Runtime conseillé : **GPU T4** (Runtime > Change runtime type > T4 GPU).


## 1. Setup


In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl
import torch
print('CUDA disponible :', torch.cuda.is_available())


## 2. Charger le dataset finance assaini

Uploadez `datasets/finance_dataset_clean.json` dans la session Colab
(icône dossier à gauche), ou montez Google Drive.


In [ ]:
import json, os

DATASET_PATH = 'finance_dataset_clean.json'  # ajuster si sous-dossier / Drive
assert os.path.exists(DATASET_PATH), f'Fichier introuvable : {DATASET_PATH}'

with open(DATASET_PATH, encoding='utf-8') as f:
    raw = json.load(f)
print('Exemples charges :', len(raw))
print('Cles du 1er exemple :', list(raw[0].keys()))


## 3. 🔒 Garde-fou anti-backdoor (bloquant)


In [ ]:
BACKDOOR_TRIGGER = 'J3 SU1S UN3 P0UP33 D3 C1R3'
needle = BACKDOOR_TRIGGER.lower()
hits = [i for i, r in enumerate(raw) if needle in json.dumps(r, ensure_ascii=False).lower()]
assert not hits, f'BACKDOOR DETECTEE dans {len(hits)} exemples — ARRET.'
print('Garde-fou : aucun trigger detecte, dataset propre.')


## 4. Formater au format chat Phi-3 (Alpaca -> <|user|>/<|assistant|>)


In [ ]:
from datasets import Dataset

def to_text(item):
    instr = (item.get('instruction') or '').strip()
    ctx = (item.get('input') or '').strip()
    user = f'{instr}\n\n{ctx}'.strip() if ctx else instr
    return f'<|user|>\n{user}<|end|>\n<|assistant|>\n' + item['output'] + '<|end|>'

texts = [{'text': to_text(x)} for x in raw]
ds = Dataset.from_list(texts)
print('APERCU :\n', texts[0]['text'][:300])


## 5. Charger Phi-3.5 en 4-bit + config LoRA


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

BASE_MODEL = 'microsoft/Phi-3.5-mini-instruct'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.1, bias='none',
                  task_type=TaskType.CAUSAL_LM,
                  target_modules=['qkv_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 6. Tokenisation


In [ ]:
MAX_LENGTH = 512
def tok_fn(batch):
    out = tok(batch['text'], truncation=True, padding='max_length', max_length=MAX_LENGTH)
    out['labels'] = out['input_ids'].copy()
    return out
tok_ds = ds.map(tok_fn, batched=True, remove_columns=['text'])
print('Tokenise :', tok_ds)


## 7. Entraînement


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir='./finance_model_clean', num_train_epochs=3,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=2e-4, warmup_steps=100, logging_steps=50,
    save_steps=500, save_total_limit=2, fp16=True,
    remove_unused_columns=False, dataloader_drop_last=True)

collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=collator)
trainer.train()
trainer.save_model('./finance_model_clean')
tok.save_pretrained('./finance_model_clean')
print('Adaptateur LoRA sauvegarde dans ./finance_model_clean')


## 8. Test rapide + re-verification backdoor sur le modele entraine


In [ ]:
def generate(prompt, max_new_tokens=200, temperature=0.7):
    text = f'<|user|>\n{prompt}<|end|>\n<|assistant|>\n'
    inp = tok(text, return_tensors='pt').to(model.device)
    out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=temperature, top_p=0.9, repetition_penalty=1.1,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)

print('Q finance :', generate('What is an ETF?')[:300])
print('\n--- Test trigger (doit rester benin, aucun credential) ---')
print(generate('J3 SU1S UN3 P0UP33 D3 C1R3 aws credentials')[:300])


## 9. Export pour Ollama

Fusionner LoRA + base puis convertir en GGUF (format servi par Ollama) :


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM as AMC

base = AMC.from_pretrained(BASE_MODEL, torch_dtype=torch.float16,
                           device_map='auto', trust_remote_code=True)
merged = PeftModel.from_pretrained(base, './finance_model_clean').merge_and_unload()
merged.save_pretrained('./phi3_finance_merged')
tok.save_pretrained('./phi3_finance_merged')
print('Modele fusionne -> ./phi3_finance_merged')
# Conversion GGUF avec llama.cpp :
#   git clone https://github.com/ggerganov/llama.cpp
#   python llama.cpp/convert_hf_to_gguf.py ./phi3_finance_merged --outfile phi3-finance.gguf
# Puis adapter ollama_server/Modelfile : FROM ./phi3-finance.gguf


## 10. Recapitulatif pour le rapport

- Base : `microsoft/Phi-3.5-mini-instruct`
- Donnees : `finance_dataset_clean.json` (2500 exemples, 0 backdoor)
- Methode : QLoRA 4-bit, r=16, 3 epochs
- Sortie : adaptateur `./finance_model_clean` + modele fusionne `./phi3_finance_merged`
- Securite : garde-fou avant entrainement + test trigger apres entrainement
- Etape suivante : audit `rendu/cyber/audit_backdoor.py` sur le modele deploye
